## Problema 2 Cuerpos

In [1]:
import pymcel as pc 
import numpy as np 
import matplotlib.pyplot as plt
import rebound as rb

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!


### Simulación con REBOUND

In [28]:
#Constantes
G = pc.constantes.G
M_sol = pc.constantes.M_sun
M_tierra = pc.constantes.M_earth
AU = pc.constantes.au
v_tierra = np.sqrt(G * M_sol / AU)

In [37]:
# 1. Configuración inicial
sim = rb.Simulation()
sim.G = G
sim.units = ('m', 's', 'kg') # Unidades SI
sim.integrator = "ias15" # Integrador de alta precisión para encuentros cercanos

# 2. Añadir cuerpos
# Sol en el origen
sim.add(m=M_sol, x=0, y=0, z=0, vx=0, vy=0, vz=0) 

# Tierra
sim.add(m=M_tierra, x=AU, vy=v_tierra)

# Apophis: Lo posicionamos para que el encuentro ocurra DESPUÉS del t=0
# Lo alejamos 2 millón de km de la Tierra, con un desplazamiento en y para simular un impacto cercano
distancia_inicial = 2.0e9 # 2 millones de km
impact_parameter = 1.08e8 

sim.add(m=2.7e10, 
        x=AU + distancia_inicial, 
        y= impact_parameter, # Desplazamiento en y para simular el impacto cercano
        vx=-5850.0,           # Velocidad relativa en x (hacia la Tierra)
        vy=v_tierra ) 

sim.move_to_com()

# 3. Integración con búsqueda de Perigeo
dias = 10
tiempos = np.linspace(0, dias * 86400, 100000) # 10 días en segundos
d_min = float('inf')
datos_perigeo = {}

for t in tiempos:
    sim.integrate(t)
    e = sim.particles[1]
    a = sim.particles[2]
    d = np.sqrt((a.x-e.x)**2 + (a.y-e.y)**2 + (a.z-e.z)**2)
    
    if d < d_min:
        d_min = d
        t_perigeo = t

print(f"Distancia mínima: {d_min:.4e} m")

Distancia mínima: 9.7310e+07 m


**REBOUND vs Leapfrog**

1. El método de Leapfrog es un integrador que es bueno conservando la energía a largo plazo de órbitas cerradas. Sin embargo, en un encuentro hiperbólico como el de Apophis, donde la velocidad cambia drásticamente en muy poco tiempo, este método puede acumular errores númericos. REBOUND, al ser de alto ordena mantiene la energía constante incluso en condiciones de influencia gravitaroria más extremas. 
2. En Leapfrog, si el paso de tiempo ($\Delta t$) es demasiado grande, el asteroide podría saltarse el punto exacto de máxima cercanía, dando un perigeo falso. En cambio REBOUND detecta el cambio en el gradiente gravitatorio y ajusta el tiempo para determinar el momento exacto del encuentro. Por eso se paso de un perigeo de $10^8$ m (Leapfrog) a uno más preciso $10^7$ m.

Mientras que el método de Leapfrog permitió una primera aproximación a la trayectoria de Apophis, la implementación de REBOUND fue necesaria para capturar la dinámica de alta energía del encuentro. Gracias al integrador IAS15, se logró modelar la transición jerárquica con una presición de orden superior, demostrando que el perigeo es extremadamente sensible a las condiciones iniciales y que solo un esquema de paso adaptativo puede garantizar la conservación de la energía en encuentros tan cercanos como el de Apophis en el 2029.

In [3]:
import rebound
import numpy as np

# --- 1. CONFIGURACIÓN INICIAL (SI) ---
sim = rebound.Simulation()
sim.G = 6.67430e-11 # m^3 kg^-1 s^-2
sim.integrator = "ias15" # Precisión de 15vo orden para encuentros cercanos

# --- 2. AÑADIR TODOS LOS OBJETOS DEL SISTEMA SOLAR ---
# Datos aproximados en SI (Masa en kg)
# En una investigación profesional, estos se obtendrían de NASA Horizons
sim.add(m=1.98847e30, name="Sun")
sim.add(m=3.3011e23,  name="Mercury")
sim.add(m=4.8675e24,  name="Venus")
sim.add(m=5.9722e24,  name="Earth")
sim.add(m=6.4171e23,  name="Mars")
sim.add(m=1.8982e27,  name="Jupiter")
sim.add(m=5.6834e26,  name="Saturn")
sim.add(m=8.6810e25,  name="Uranus")
sim.add(m=1.0241e26,  name="Neptune")

# --- 3. AÑADIR A APOPHIS ---
# Usamos tus parámetros de investigación para el encuentro
dist_x = 2.0e9  
impact_parameter = 1.08e8  
v_infinito = 5850.0        

# Posicionamos a Apophis relativo a la Tierra (sim.particles[3])
tierra = sim.particles[3]
sim.add(m=2.7e10, name="Apophis",
        x = tierra.x + dist_x, 
        y = tierra.y + impact_parameter, 
        vx = tierra.vx - v_infinito, 
        vy = tierra.vy)

sim.move_to_com() # Mover al centro de masa del Sistema Solar

# --- 4. INTEGRACIÓN Y BÚSQUEDA DEL PERIGEO ---
tiempos = np.linspace(0, 10 * 86400, 100000) 
distancia_minima = float('inf')

for t in tiempos:
    sim.integrate(t)
    # Calculamos la distancia específica Apophis-Tierra
    dx = sim.particles["Apophis"].x - sim.particles["Earth"].x
    dy = sim.particles["Apophis"].y - sim.particles["Earth"].y
    dz = sim.particles["Apophis"].z - sim.particles["Earth"].z
    dist_actual = np.sqrt(dx**2 + dy**2 + dz**2)
    
    if dist_actual < distancia_minima:
        distancia_minima = dist_actual

# --- 5. RESULTADOS EN NOTACIÓN CIENTÍFICA ---
print(f"--- SIMULACIÓN SISTEMA SOLAR COMPLETO ---")
print(f"Distancia mínima Apophis-Tierra: {distancia_minima:.4e} metros")

TypeError: Particle.__init__() got an unexpected keyword argument 'name'